In [21]:
import os
import re
import logging
import multiprocessing
from typing import List, Dict, Any

import pandas as pd
import numpy as np
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression, Perceptron
from sklearn.ensemble import RandomForestClassifier

from gensim.models.word2vec import Word2Vec
from gensim.models.doc2vec import TaggedDocument, Doc2Vec

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Constants and configuration
CORES = multiprocessing.cpu_count()
STOPWORDS = list(stopwords.words('english')) + [
    "'d", "'ll", "'re", "'s", "'ve", '``', 'could', 'might', 'must', "n't", 'need', 'sha', 'wo', 'would', 'use', 
]
    
DATA_FOLDER = '../data/training_datasets'
TEST_DATA_FILE = '../data/test_dataset_10.xlsx'
RESULTS_FOLDER = '../results'
N_FEATURES_VALUES = [5000, 10000, 15000]

ALGORITHMS = {
    'Multinomial Naive Bayes' : MultinomialNB(),
    'Support Vector Machines (SVM)': LinearSVC(dual="auto"),
    'Logistic Regression': LogisticRegression(n_jobs=CORES, random_state=42),
}

# Function definitions
def load_datasets(folder: str) -> List[str]:
    return os.listdir(folder)

def load_test_data(filepath: str) -> pd.DataFrame:
    return pd.read_excel(filepath)

def tokenize_remove_stop_words(text: str) -> List[str]:
    return [token for token in word_tokenize(text) if 
            token not in STOPWORDS and
            len(token) > 2 and  # Mots de moins de 2 lettres
            not (bool(re.search(r'\d', token))) and # Mots contenant des chiffres
            not (any(char in string.punctuation for char in token)) # Mots contenant des signes de ponctuation
]

def evaluate_algorithm(algorithm, X, y, cv) -> List[Dict[str, Any]]:
    metrics = []
    accuracy_list = []
    precision_list = []
    recall_list = []
    f1_list = []
    
    for train_idx, test_idx in cv.split(X, y):
        algorithm.fit(X[train_idx], y[train_idx])
        y_pred = algorithm.predict(X[test_idx])
        accuracy_list.append(accuracy_score(y[test_idx], y_pred))
        precision_list.append(precision_score(y[test_idx], y_pred, average=None, zero_division=0))
        recall_list.append(recall_score(y[test_idx], y_pred, average=None, zero_division=0))
        f1_list.append(f1_score(y[test_idx], y_pred, average=None, zero_division=0))
    
    # Calculer les moyennes pour chaque pli et stocker les résultats
    avg_accuracy = np.mean(accuracy_list)
    avg_precision = np.mean(precision_list, axis=0)
    avg_recall = np.mean(recall_list, axis=0)
    avg_f1 = np.mean(f1_list, axis=0)
    
    for class_index, class_label in enumerate(algorithm.classes_):
        metrics.append({
            'class': class_label,
            'accuracy': avg_accuracy,
            'precision': avg_precision[class_index],
            'recall': avg_recall[class_index],
            'f1-score': avg_f1[class_index]
        })
    
    return metrics

def save_report(df: pd.DataFrame, filename: str):
    df.to_csv(filename, index=False)

def vectorize_tf_idf(corpus: List[str], corpus_test: List[str], n_features: int):
    vectorizer = TfidfVectorizer(
        tokenizer=tokenize_remove_stop_words,
        token_pattern=None,
        stop_words=STOPWORDS,
        max_features=n_features
    )
    X = vectorizer.fit_transform(corpus)
    X_test = vectorizer.transform(corpus_test)
    return X, X_test

datasets = load_datasets(DATA_FOLDER)
df_test = load_test_data(TEST_DATA_FILE)
corpus_test = df_test['text_post'].astype('str')
y_test = df_test['category'].astype('str')

training_reports = []
test_reports = []

for dataset in datasets[2:6]:
    ratio = int(dataset[14:-7])
    df = pd.read_excel(os.path.join(DATA_FOLDER, dataset))
    report = []

    corpus = df['text_post'].astype('str')
    y = df['category'].astype('str')

    for n_features in N_FEATURES_VALUES:
        X, X_test = vectorize_tf_idf(corpus, corpus_test, n_features)
        vectorization_label = 'TF-IDF'        

        for algorithm_name, algorithm in ALGORITHMS.items():
            metrics = evaluate_algorithm(algorithm, X, y, StratifiedKFold(shuffle=True, random_state=42))

            for metric in metrics:
                results = {
                    '% Incels': ratio,
                    'Algorithme': algorithm_name,
                    'Modèle de vectorisation': vectorization_label,
                    'N traits discr.': n_features,
                    'class': metric['class'],
                    'accuracy': metric['accuracy'],
                    'precision': metric['precision'],
                    'recall': metric['recall'],
                    'f1-score': metric['f1-score']
                }

                report.append(results)
                logger.info(f"{algorithm_name} | {ratio}% Incels | {n_features} traits discr.\n"
                            f"Class: {metric['class']}, Accuracy: {results['accuracy']:.4f}, Precision: {results['precision']:.4f}, "
                            f"Recall: {results['recall']:.4f}, F1-score: {results['f1-score']:.4f}")

            algorithm.fit(X, y)
            predictions_test = algorithm.predict(X_test)
            test_accuracy = accuracy_score(y_test, predictions_test)
            test_precision = precision_score(y_test, predictions_test, average=None, zero_division=0)
            test_recall = recall_score(y_test, predictions_test, average=None, zero_division=0)
            test_f1 = f1_score(y_test, predictions_test, average=None, zero_division=0)

            for class_index, class_label in enumerate(algorithm.classes_):
                test_results = {
                    '% Incels': ratio,
                    'Algorithme': algorithm_name,
                    'Modèle de vectorisation': vectorization_label,
                    'N traits discr.': n_features,
                    'class': class_label,
                    'accuracy': test_accuracy,
                    'precision': test_precision[class_index],
                    'recall': test_recall[class_index],
                    'f1-score': test_f1[class_index]
                }

                test_reports.append(test_results)

    report_df = pd.DataFrame(report)
    report_df['nb_posts_incels'] = (report_df['% Incels'].apply(lambda x: x / 100) * df.shape[0]).astype(int)
    report_df['nb_posts_neutral'] = (report_df['% Incels'].apply(lambda x: 1 - (x / 100)) * df.shape[0]).astype(int)
    report_df['nb_posts_total'] = df.shape[0]

    report_df = report_df[['nb_posts_total', 'nb_posts_incels', 'nb_posts_neutral', '% Incels', 'Algorithme',
                            'Modèle de vectorisation', 'N traits discr.', 'class', 'accuracy', 'precision', 'recall', 'f1-score']]

    training_reports.append(report_df)

final_report_df = pd.concat(training_reports)
save_report(final_report_df.sort_values(by='f1-score', ascending=False), os.path.join(RESULTS_FOLDER, f'results_training_top20_for-each-class.csv'))

test_report_df = pd.DataFrame(test_reports)
save_report(test_report_df.sort_values(by='f1-score', ascending=False), os.path.join(RESULTS_FOLDER, f'results_test_top20_for-each-class.csv'))

INFO:__main__:Multinomial Naive Bayes | 30% Incels | 5000 traits discr.
Class: incel, Accuracy: 0.8470, Precision: 0.8364, Recall: 0.6092, F1-score: 0.7049
INFO:__main__:Multinomial Naive Bayes | 30% Incels | 5000 traits discr.
Class: neutral, Accuracy: 0.8470, Precision: 0.8500, Recall: 0.9489, F1-score: 0.8967
INFO:__main__:Support Vector Machines (SVM) | 30% Incels | 5000 traits discr.
Class: incel, Accuracy: 0.8422, Precision: 0.7972, Recall: 0.6360, F1-score: 0.7074
INFO:__main__:Support Vector Machines (SVM) | 30% Incels | 5000 traits discr.
Class: neutral, Accuracy: 0.8422, Precision: 0.8564, Recall: 0.9306, F1-score: 0.8920
INFO:__main__:Logistic Regression | 30% Incels | 5000 traits discr.
Class: incel, Accuracy: 0.8463, Precision: 0.8437, Recall: 0.5987, F1-score: 0.7003
INFO:__main__:Logistic Regression | 30% Incels | 5000 traits discr.
Class: neutral, Accuracy: 0.8463, Precision: 0.8470, Recall: 0.9524, F1-score: 0.8966
INFO:__main__:Multinomial Naive Bayes | 30% Incels | 1